# RoadGuard — Data Preparation

Prepare the ANOVOX Town04 Static and Normality Mono datasets
for anomaly segmentation.

This notebook orchestrates archive extraction, metadata processing,
mask generation, manifest validation, and reproducible dataset splitting.

All reusable processing logic resides in the roadguard Python package.

## 1. Connect Google Drive

Mount Google Drive to access the original ANOVOX archives and store the prepared dataset persistently.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Set up the RoadGuard environment

Clone the project repository and install its Python package in the Colab runtime. If installation requires a runtime restart, reconnect Drive and restore the working directory and imports as needed. Do not clone again if the repository is already present in the runtime.

In [ ]:
%cd /content
!git clone https://github.com/berns099/road-guard.git
%cd /content/road-guard

In [ ]:
%pip install -e .

## 3. Configure data locations

Import the data-preparation utilities and define the source and output folders. `RoadGuard_Dataset` contains the original archives; `RoadGuard_Processed` stores extracted data, manifests, splits, and the final package. Both folders must be accessible under the mounted `MyDrive` path.

In [ ]:
from pathlib import Path

from roadguard.data.preparation import (
    extract_anovox_archive,
    load_scenario_metadata,
    generate_anomaly_masks,
)

from roadguard.data.manifest import (
    build_master_manifest,
    validate_manifest,
)

# Folder containing the original ANOVOX archives
DATA_DIR = Path("/content/drive/MyDrive/RoadGuard_Dataset")

# Persistent folder for prepared data and experiment splits
OUTPUT_DIR = Path("/content/drive/MyDrive/RoadGuard_Processed")

EXTRACT_DIR = OUTPUT_DIR / "extracted"
PROCESSED_DIR = OUTPUT_DIR / "processed"

## 4. Extract the ANOVOX archives

Extract the Town04 Static and Normality Mono archives, retaining the RGB, depth, semantic, and scenario configuration files needed for segmentation.

In [ ]:
STATIC_ARCHIVE = DATA_DIR / "AnoVox_Static_Mono_Town04.tar.gz"
NORMALITY_ARCHIVE = DATA_DIR / "AnoVox_Normality_Mono_Town04.tar.gz"

static_counts = extract_anovox_archive(
    archive_path=STATIC_ARCHIVE,
    destination=EXTRACT_DIR,
)

normality_counts = extract_anovox_archive(
    archive_path=NORMALITY_ARCHIVE,
    destination=EXTRACT_DIR,
)

print("Static:", static_counts)
print("Normality:", normality_counts)

## 5. Load scenario metadata

Read the configuration maps for both datasets and combine scenario-level metadata into one table.

In [ ]:
import pandas as pd

STATIC_CONFIG = (
    EXTRACT_DIR
    / "AnoVox_Static_Mono_Town04"
    / "Scenario_Configuration_Files"
    / "scenario_config_map_Town04.json"
)

NORMALITY_CONFIG = (
    EXTRACT_DIR
    / "AnoVox_Normality_Mono_Town04"
    / "Scenario_Configuration_Files"
    / "scenario_config_map_Town04.json"
)

static_metadata = load_scenario_metadata(STATIC_CONFIG, source="static")
normality_metadata = load_scenario_metadata(NORMALITY_CONFIG, source="normality")

scenario_metadata = pd.concat(
    [static_metadata, normality_metadata],
    ignore_index=True,
)

print(f"Loaded {len(scenario_metadata)} scenarios")
print(scenario_metadata.groupby("source").size())

## 6. Generate anomaly masks

Generate binary anomaly masks from semantic labels. Existing valid masks are reused when `overwrite=False`.

In [ ]:
mask_metadata = generate_anomaly_masks(
    data_root=EXTRACT_DIR,
    overwrite=False,
)

print(f"Processed {len(mask_metadata)} masks")
print(f"Frames containing anomaly pixels: {(mask_metadata['anomaly_pixels'] > 0).sum()}")

## 7. Build the master manifest

Associate each frame with its RGB image, depth map, semantic label, anomaly mask, and scenario metadata. Image paths are stored relative to the extracted dataset root.

In [ ]:
manifest = build_master_manifest(
    data_root=EXTRACT_DIR,
    scenario_metadata=scenario_metadata,
    mask_metadata=mask_metadata,
)

print(f"Manifest frames: {len(manifest)}")
print(manifest.groupby("source").size())

## 8. Validate the manifest

Check file availability, image and mask dimensions, mask values, anomaly-pixel counts, and consistency between the manifest and extracted files. This may take time on mounted Drive storage.

In [ ]:
validate_manifest(
    manifest=manifest,
    data_root=EXTRACT_DIR,
)

print("Master manifest validation passed.")

## 9. Save the master manifest

Persist the validated frame index for reuse in training and evaluation notebooks.

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = PROCESSED_DIR / "master_manifest.csv"
manifest.to_csv(manifest_path, index=False)

print(f"Saved {len(manifest)} frames to {manifest_path}")

## 10. Create reproducible dataset splits

Split Normality scenarios once into fixed train, validation, and test partitions. Separately, create leave-one-family-out (LOFO) splits for Static anomaly scenarios: each fold holds out one anomaly family for testing. Combine the fixed Normality assignments with each Static fold so the Normality test set remains consistent across folds.

In [ ]:
from roadguard.data.splits import create_standard_split, create_lofo_splits

normality_manifest = manifest[manifest["source"] == "normality"].copy()
static_manifest = manifest[manifest["source"] == "static"].copy()

normality_splits = create_standard_split(
    normality_manifest,
    seed=42,
)

static_folds = create_lofo_splits(
    static_manifest,
    seed=42,
)

folds = {
    family: pd.concat(
        [static_fold, normality_splits],
        ignore_index=True,
    )
    for family, static_fold in static_folds.items()
}

print(f"Generated {len(folds)} LOFO folds")
print("Held-out anomaly families:", list(folds))

## 11. Save split assignments

Save the fixed Normality split and one complete frame-level CSV per LOFO fold. These files define the exact partitions used by downstream experiments.

In [ ]:
SPLITS_DIR = PROCESSED_DIR / "lofo_splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

normality_splits.to_csv(
    PROCESSED_DIR / "normality_splits.csv",
    index=False,
)

for family, fold in folds.items():
    fold.to_csv(
        SPLITS_DIR / f"{family}.csv",
        index=False,
    )

print(f"Saved {len(folds)} LOFO folds to {SPLITS_DIR}")

## 12. Package the prepared dataset

Create a single uncompressed TAR archive containing both the extracted dataset and the processed manifest/split CSVs. This makes it easier to stage the complete dataset on a training runtime without copying many individual files from Drive. The original files remain available separately.

In [ ]:
import tarfile

archive_path = OUTPUT_DIR / "roadguard_prepared.tar"

with tarfile.open(archive_path, "w") as tar:
    tar.add(EXTRACT_DIR, arcname="extracted")
    tar.add(PROCESSED_DIR, arcname="processed")

print(f"Package created: {archive_path}")
print(f"Package size: {archive_path.stat().st_size / (1024**3):.2f} GB")